# 02 — Обучение моделей

Читает артефакт из `data/demo_run`, сохраняет веса в `models/<run_id>/`.

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.nn import (
    DPIFlow,
    EVTNeuralSSM,
    GRUBaseline,
    RiskMLP,
    TCNBaseline,
    train_model,
)

artifact_dir = Path("data/demo_run")
population, config = load_population_artifact(artifact_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

benchmark_data = prepare_benchmark_dataset(population, config, device)

static_dim = benchmark_data["train"]["static"].shape[1]
prefix_dim = benchmark_data["train"]["prefix_summary"].shape[1]
seq_dim = benchmark_data["train"]["seq_in"].shape[-1]

run_id = "demo_run"
out_dir = Path("models") / run_id
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "config.json").write_text(json.dumps(asdict(config), indent=2), encoding="utf-8")
print("device:", device, "сохранение в", out_dir.resolve())

In [ ]:
models_to_train = [
    ("mlp_risk", lambda: RiskMLP(static_dim=static_dim, prefix_dim=prefix_dim, hidden_dim=128)),
    ("gru", lambda: GRUBaseline(static_dim=static_dim, seq_dim=seq_dim, hidden_dim=96)),
    ("tcn", lambda: TCNBaseline(static_dim=static_dim, seq_dim=seq_dim, hidden_dim=96)),
    (
        "dpi_flow",
        lambda: DPIFlow(
            static_dim=static_dim,
            prefix_dim=prefix_dim,
            seq_len=config.seq_len,
            prefix_len=config.prefix_len,
            max_cycle_reference=config.max_cycle_reference,
            theta_dim=31,
            hidden_dim=160,
            probabilistic=True,
            calibration_steps=2,
            calibration_lr=0.10,
            use_analytical_layer=True,
        ),
    ),
    (
        "evt_ssm",
        lambda: EVTNeuralSSM(
            static_dim=static_dim,
            prefix_dim=prefix_dim,
            seq_dim=seq_dim,
            seq_len=config.seq_len,
            prefix_len=config.prefix_len,
            max_cycle_reference=config.max_cycle_reference,
            hidden_dim=144,
            use_trigger_head=True,
            structured_post_event=True,
            use_crr_damage=True,
        ),
    ),
]

epoch_map = {
    "mlp_risk": config.baseline_epochs,
    "gru": config.baseline_epochs,
    "tcn": config.baseline_epochs,
    "dpi_flow": config.physics_epochs,
    "evt_ssm": config.physics_epochs,
}

for name, factory in models_to_train:
    model = factory().to(device)
    epochs = epoch_map[name]
    model, history = train_model(
        model,
        benchmark_data["train"],
        benchmark_data["val"],
        epochs=epochs,
        model_name=name,
        config=config,
        device=device,
    )
    torch.save(model.state_dict(), out_dir / f"{name}.pt")
    history.to_parquet(out_dir / f"history_{name}.parquet")
    print("OK", name)